# Business solution Project
## Challenge
Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

In [35]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI # If you will be using OPENAI API
from google import genai

In [47]:
# Initialize and constants

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY in your .env file.")

MODEL = "gemini-3.5-flash"
client = genai.Client(api_key=api_key)

In [ ]:
response = client.interactions.create(
    model=MODEL,
    input="What is AI",
)
print(response.output_text)

In [ ]:
links = fetch_website_links("https://edwarddonner.com/")
links

In [41]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [42]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [44]:
# Define the schema once
LINKS_SCHEMA = {
    "type": "object",
    "properties": {
        "links": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "url": {"type": "string"},
                },
                "required": ["type", "url"],
            },
        }
    },
    "required": ["links"],
}

In [45]:
def select_relevant_links(url: str) -> dict:
    response = client.interactions.create(
        model=MODEL,
        input=f"{link_system_prompt}\n\n{get_links_user_prompt(url)}",
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": LINKS_SCHEMA,
        },
    )
    return json.loads(response.output_text)

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

### Same Function with 2 extra Print statement, for clear Understanding while showing output/Result

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.interactions.create(
        model=MODEL,
        input=f"{link_system_prompt}\n\n{get_links_user_prompt(url)}",
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": LINKS_SCHEMA,
        },
    )
    links = json.loads(response.output_text)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to gemini-3.5-flash

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

### So far we've wrapped an Existing Gemini that we already made, next step is to turn this into our second call to Gemini

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("edward Donner", "https://huggingface.co")

In [21]:
def create_brochure(company_name, url):
    response = client.interactions.create(
        model=MODEL,
        system_instruction=brochure_system_prompt,
        input=get_brochure_user_prompt(company_name, url),
    )
    
    result = response.output_text
    display(Markdown(result))

In [ ]:
create_brochure("Edward Donner", "https://edwarddonner.com")

## Stream Method: Which will output the Result Token By Token as a typewritter

In [ ]:
def stream_brochure(company_name, url):
    stream = client.models.generate_content_stream(
        model=MODEL,
        contents=get_brochure_user_prompt(company_name, url),
        config=genai.types.GenerateContentConfig(
            system_instruction=brochure_system_prompt
        ),
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.text or ""
        update_display(Markdown(response), display_id=display_handle.display_id)


stream_brochure("Edward Donner", "https://edwarddonner.com")